### React Agent and components of Langgraph

In [20]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

In [21]:
def mul(a:int,b:int) -> int:

    """ Multiply a and b
    Args:
    a: first int
    b: Second int
    """
    return a*b
def add(a:int,b:int) -> int:

    """ ADD a and b
    Args:
    a: first int
    b: Second int
    """
    return a+b
def div(a:int,b:int) -> float:

    """ Divide a and b
    Args:
    a: first int
    b: Second int
    """
    return a/b

In [22]:
llm = ChatGroq(model = "qwen/qwen3-32b",api_key = os.getenv("GROQ_API_KEY"))
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A7F3BA2D70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A7F3BA23F0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [23]:
tools= [add,mul,div]
# Bind the tool
llm_with_tools = llm.bind_tools(tools,parallel_tool_calls = False)

In [24]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
# create a Prompt
system_message = SystemMessage(content = " you are a expert in Arthematic operations")

In [25]:
def assistant(state:MessagesState):
    return{"messages":[llm_with_tools.invoke([system_message]+state["messages"])]}

In [26]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode   # tool calls
from langgraph.prebuilt import tools_condition
from IPython.display import Markdown

builder = StateGraph(MessagesState)
builder.add_node("assistant",assistant)
builder.add_node("tools",ToolNode(tools))

builder.add_edge(START,"assistant")
builder.add_conditional_edges("assistant",tools_condition)
builder.add_edge("tools","assistant")

graph = builder.compile()
graph.get_graph().print_ascii()
# display(Markdown(f```text\n{graph.get_graph}))



        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
        +-----------+         
        | assistant |         
        +-----------+         
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


In [27]:
messages = graph.invoke

In [28]:
messages = [HumanMessage(content = "Add 3 and 4 . Mul by output by 2. Div the output by 5")]
messages = graph.invoke({"messages":messages})

for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Add 3 and 4 . Mul by output by 2. Div the output by 5
================================== Ai Message ==================================
Tool Calls:
  add (yxzrxpa47)
 Call ID: yxzrxpa47
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: add

7
================================== Ai Message ==================================
Tool Calls:
  mul (dgy6ebx58)
 Call ID: dgy6ebx58
  Args:
    a: 7
    b: 2
================================= Tool Message =================================
Name: mul

14
================================== Ai Message ==================================
Tool Calls:
  div (qgxy99yrp)
 Call ID: qgxy99yrp
  Args:
    a: 14
    b: 5
================================= Tool Message =================================
Name: div

2.8
================================== Ai Message ==================================

The final result after pe

### Memory Saver

## Streaming
### In streaming there are two modes 1. update(only updated messages state will update) 2. value(Full Messages)

In [38]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from typing import Annotated #for list of messages annotated will give the metadate info
from langgraph.graph.message import add_messages
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

class state(TypedDict):
    messages:Annotated[list,add_messages]

In [30]:
llm = ChatGroq(model = "qwen/qwen3-32b",api_key = os.getenv("GROQ_API_KEY"))
llm.invoke("Hello")

AIMessage(content='<think>\nOkay, the user said "Hello". That\'s a greeting. I should respond in a friendly and welcoming way. Let me make sure to keep it open-ended so they feel comfortable to ask anything. Maybe add an emoji to keep it approachable. Let me check for any typos. Yeah, that looks good. Ready to respond.\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 9, 'total_tokens': 93, 'completion_time': 0.16648397, 'completion_tokens_details': None, 'prompt_time': 0.000361704, 'prompt_tokens_details': None, 'queue_time': 0.052797936, 'total_time': 0.166845674}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db927-6850-7cf2-a8c4-9c4f529aedc9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 84, 'total_t

In [ ]:
from langgraph.checkpoint.memory import MemorySaver # MemorySaver is easiest way of checkpoint
memory =MemorySaver()

def chatbot(state:MessagesState):
    return{"messages":[llm.invoke(state['messages'])]}
builder = StateGraph(MessagesState)

builder.add_node("chatbot",chatbot)

builder.add_edge(START,"chatbot")
builder.add_edge("chatbot",END)

graph = builder.compile(checkpointer=memory)

config ={"configurable":{"thread_id":"1"}}

graph.invoke({"messages":"My Name is king I like Kingdom"},config)

{'messages': [HumanMessage(content='My Name is king I like Kingdom', additional_kwargs={}, response_metadata={}, id='f9c67c80-8c53-4ebe-833f-aac46283ff63'),
  AIMessage(content='<think>\nOkay, the user mentioned their name is King and they like kingdoms. Let me start by acknowledging their name and showing interest in their interest. Maybe they\'re into history, games, or fantasy. I should ask clarifying questions to find out more about what they enjoy about kingdoms. Are they into historical kingdoms, fictional ones in books or games, or maybe building their own? I need to open up the conversation to explore their specific interests. Also, since they mentioned "Kingdom," maybe they\'re into role-playing or have a creative project in mind. I should encourage them to elaborate so I can provide relevant information or suggestions. Let me make sure my response is friendly and engaging, inviting them to share more details.\n</think>\n\nHello, King! That\'s a mighty name to go with a love f

In [ ]:
# Stream mode using updates where it will give the updated messages
for chunk in graph.stream({"messages": " Hi My Name is King I like Travelling"},config,stream_mode="updates"):
    print(chunk)

{'chatbot': {'messages': [AIMessage(content="<think>\nOkay, the user is King and mentioned he likes traveling. Let me start by acknowledging his name and his interest. He previously mentioned liking kingdoms, so maybe connecting travel with that.\n\nHe might be interested in real-world travel or maybe fictional journeys. I should ask for clarification. Travel can mean different things: exploring new places, adventure, maybe even through books or games.\n\nI should ask if he's into historical travels, like exploring ancient kingdoms, or maybe modern travel. Also, if he's into fictional travel through books or movies. Maybe he's into planning trips or virtual travel. Need to prompt him to specify so I can engage better.\n\nI can mention examples like historical routes, famous travelers, or fictional adventures. That way, he can pick what interests him. Keep the tone friendly and encouraging.\n</think>\n\nHello again, King! 🌍✨ Your love for traveling opens up a world of possibilities—do y

In [ ]:
# Stream mode using values where it will give the full messages it will includes the previous messges

for chunk in graph.stream({"messages": " Hi My Name is King I like Travelling"},config,stream_mode="values"):
    print(chunk)

{'messages': [HumanMessage(content='My Name is king I like Kingdom', additional_kwargs={}, response_metadata={}, id='f9c67c80-8c53-4ebe-833f-aac46283ff63'), AIMessage(content='<think>\nOkay, the user mentioned their name is King and they like kingdoms. Let me start by acknowledging their name and showing interest in their interest. Maybe they\'re into history, games, or fantasy. I should ask clarifying questions to find out more about what they enjoy about kingdoms. Are they into historical kingdoms, fictional ones in books or games, or maybe building their own? I need to open up the conversation to explore their specific interests. Also, since they mentioned "Kingdom," maybe they\'re into role-playing or have a creative project in mind. I should encourage them to elaborate so I can provide relevant information or suggestions. Let me make sure my response is friendly and engaging, inviting them to share more details.\n</think>\n\nHello, King! That\'s a mighty name to go with a love for